In [1]:
feedback_data = [
    {
        "customer": "Jordan",
        "feedback": "Everything arrived early and the service was excellent."
    },
    {
        "customer": "Maya",
        "feedback": "I've contacted support three times and nobody has gotten back to me."
    },
    {
        "customer": "Alex",
        "feedback": "There are purchases on my card that I don't recognize."
    },
    {
        "customer": "Taylor",
        "feedback": "The app keeps freezing whenever I try to check out."
    },
    {
        "customer": "Morgan",
        "feedback": "I was billed after cancelling my subscription last month."
    },
    {
        "customer": "Sam",
        "feedback": "The product works fine but setup was more confusing than I expected."
    }
]

print(f"Customer feedback records loaded: {len(feedback_data)}")

Customer feedback records loaded: 6


In [3]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

In [4]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

print("OpenAI client ready!")

OpenAI client ready!


In [5]:
alex_feedback = feedback_data[2]["feedback"]

prompt = f"""
Analyze the following customer feedback.

Customer feedback:
{alex_feedback}

Return:
- Sentiment
- Category
- Severity
- Short summary
- Recommended action
"""

response = client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

print(response.output_text)

- **Sentiment:** Negative, worried
- **Category:** Fraud / Unauthorized transaction
- **Severity:** High
- **Short summary:** The customer reports unrecognized purchases on their card, which may indicate unauthorized activity.
- **Recommended action:** Advise the customer to review recent transactions immediately, freeze or block the card if needed, and contact fraud support to dispute the charges and investigate potential card compromise.


In [6]:
import json

prompt = f"""
Analyze this customer feedback:

{alex_feedback}

Return ONLY valid JSON using exactly these fields:

{{
    "sentiment": "positive, neutral, or negative",
    "category": "short category name",
    "severity": "low, medium, or high",
    "summary": "one sentence summary",
    "recommended_action": "one sentence recommended action"
}}

Do not include markdown or any text outside the JSON.
"""

response = client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

analysis = json.loads(response.output_text)

print(analysis)

{'sentiment': 'negative', 'category': 'unauthorized charges', 'severity': 'high', 'summary': 'The customer reports unrecognized purchases on their card, suggesting possible unauthorized transactions.', 'recommended_action': 'Immediately investigate the transactions, secure the account if needed, and guide the customer through the dispute or fraud process.'}


In [7]:
all_analyses = []

for record in feedback_data:

    customer = record["customer"]
    feedback = record["feedback"]

    prompt = f"""
    Analyze this customer feedback:

    {feedback}

    Return ONLY valid JSON using exactly these fields:

    {{
        "sentiment": "positive, neutral, or negative",
        "category": "short category name",
        "severity": "low, medium, or high",
        "summary": "one sentence summary",
        "recommended_action": "one sentence recommended action"
    }}

    Do not include markdown or any text outside the JSON.
    """

    response = client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )

    analysis = json.loads(response.output_text)

    # Add customer information
    analysis["customer"] = customer
    analysis["original_feedback"] = feedback

    # Store the completed analysis
    all_analyses.append(analysis)

    print(
        f"{customer}: "
        f"{analysis['sentiment'].upper()} | "
        f"{analysis['severity'].upper()} | "
        f"{analysis['category']}"
    )

Jordan: POSITIVE | LOW | shipping
Maya: NEGATIVE | HIGH | support responsiveness
Alex: NEGATIVE | HIGH | fraud
Taylor: NEGATIVE | HIGH | checkout issue
Morgan: NEGATIVE | HIGH | billing issue
Sam: NEUTRAL | LOW | setup


In [8]:
high_priority = []

for analysis in all_analyses:
    if analysis["severity"].lower() == "high":
        high_priority.append(analysis)

print(f"High-priority cases detected: {len(high_priority)}")

for case in high_priority:
    print(
        f"{case['customer']}: "
        f"{case['category']} | "
        f"{case['severity'].upper()}"
    )

High-priority cases detected: 4
Maya: support responsiveness | HIGH
Alex: fraud | HIGH
Taylor: checkout issue | HIGH
Morgan: billing issue | HIGH


In [9]:
report = """
AI CUSTOMER FEEDBACK INTELLIGENCE REPORT
========================================

"""

report += f"Total feedback analyzed: {len(all_analyses)}\n"
report += f"High-priority cases: {len(high_priority)}\n\n"

for analysis in all_analyses:

    report += f"Customer: {analysis['customer']}\n"
    report += f"Sentiment: {analysis['sentiment'].upper()}\n"
    report += f"Category: {analysis['category']}\n"
    report += f"Severity: {analysis['severity'].upper()}\n"
    report += f"Summary: {analysis['summary']}\n"
    report += f"Recommended Action: {analysis['recommended_action']}\n"
    report += "-" * 60 + "\n"

with open("ai_feedback_report.txt", "w") as file:
    file.write(report)

print(report)

print("✅ Report saved to ai_feedback_report.txt")


AI CUSTOMER FEEDBACK INTELLIGENCE REPORT

Total feedback analyzed: 6
High-priority cases: 4

Customer: Jordan
Sentiment: POSITIVE
Category: shipping
Severity: LOW
Summary: The customer reports that the order arrived early and the service was excellent.
Recommended Action: Acknowledge the positive experience and encourage the customer to share additional feedback or shop again.
------------------------------------------------------------
Customer: Maya
Sentiment: NEGATIVE
Category: support responsiveness
Severity: HIGH
Summary: The customer reports repeated contact attempts to support without receiving any response.
Recommended Action: Have support urgently follow up with the customer, acknowledge the missed messages, and resolve the issue promptly.
------------------------------------------------------------
Customer: Alex
Sentiment: NEGATIVE
Category: fraud
Severity: HIGH
Summary: The customer reports unrecognized purchases on their card, indicating potential unauthorized activity.
R